# Phase 04 v2 — headline → forward-return regressionFine-tunes Qwen2.5-1.5B-Instruct with LoRA to predict a stock's **5-dayexcess return** from its recent headlines.## Why this target, and not the one v1 usedv1 trained a model to imitate GPT-4o-mini's BUY/HOLD/SELL opinion. Itfailed twice, for one reason: the teacher said HOLD on 96.8% of calls, sothe constant function was the loss minimum and both adapters found it. Anablation had separately shown that same opinion was making the tradingsystem *worse* — so even a perfect imitation would have imitated somethingworth deleting.Here the labels come from price data. They are free, continuous, and havereal variance. None of the three failure modes apply.## The bar, set before training`scripts/tfidf_baseline.py` was run first. On the only leak-free split —unseen symbols **and** unseen dates — TF-IDF plus ridge scored:| metric | value ||---|---|| R² | −0.182 (worse than a constant) || Spearman IC | **+0.002** || sign accuracy | 51.3% vs a 52.4% base rate |**Ship it if this model reaches IC ≥ 0.03 on that split. Otherwise reportand stop.** Written down now, because choosing what counts as successafter seeing the result is how people talk themselves into noise.One caution carried over from v1: a low MAE means nothing on its own here.The labels are z-scored and centred, so predicting 0.0 for every rowalready scores well. Every metric below is printed next to what a constantpredictor achieves.

## 1. Single GPU, before anything imports torchKaggle's T4 x2 makes `Trainer` wrap the model in `DataParallel`, which on a4-bit quantised model reliably ends in `CUDA error: an illegal memoryaccess`. `device_map={"": 0}` does not prevent it — that only pins whereweights load; Trainer counts visible devices itself.This cell must run first. After an illegal memory access the CUDA contextis dead and nothing works until a real kernel restart.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
print("visible GPUs:", torch.cuda.device_count())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets

## 2. Load the splitsUpload `data/return_dataset/` (three .jsonl files) as a Kaggle dataset andpoint `DATA` at it.`val_clean` is the split that decides the outcome: unseen symbols ANDunseen dates. The other two are diagnostic.`val_symbol` on its own is **not** trustworthy and is kept only to show why— it holds out tickers but spans the same date range as training, so amodel can score on it by learning market regimes rather than headlines. Onthe TF-IDF baseline that gap was IC +0.069 versus +0.002.

In [ ]:
import json, random
from pathlib import Path

DATA = Path("/kaggle/input/pta-return-dataset")   # <- adjust to your dataset slug
VAL_START = "2023-08-01"                          # must match build_return_dataset.py

def load(name, min_day=None):
    rows = []
    with (DATA / name).open(encoding="utf-8") as fh:
        for line in fh:
            r = json.loads(line)
            if min_day and r["day"] < min_day:
                continue
            rows.append(r)
    return rows

train      = load("train.jsonl")
val_time   = load("val_time.jsonl")
val_symbol = load("val_symbol.jsonl")
val_clean  = load("val_symbol.jsonl", VAL_START)

for name, rows in [("train", train), ("val_time", val_time),
                   ("val_symbol (leaky)", val_symbol), ("val_clean", val_clean)]:
    print(f"{name:<20} {len(rows):>7,}")

print()
print("example:", json.dumps(random.choice(train), indent=1)[:400])

## 3. Tokeniser and model`AutoModelForSequenceClassification` with `num_labels=1` and`problem_type="regression"` — a real regression head optimising MSE,rather than asking a generative model to emit a number as text. Thegenerative route was tried in v1 and brings its own failure modes(degenerate repetition under gradient checkpointing, JSON truncation);none of them exist here.Two things that silently break this setup if missed:- **Qwen has no pad token.** Both the tokeniser and `model.config` need one  set, because a sequence-classification head on a decoder model finds the  final real token by looking for `pad_token_id`. Leave it unset and it  reads padding as content.- **`modules_to_save=["score"]`.** The classification head is randomly  initialised. LoRA adapters alone would leave it random forever and the  model would learn nothing while training loss looked plausible.

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          BitsAndBytesConfig)

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_LEN = 768   # ~25 headlines fit comfortably; the dataset caps at 25.

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=1,
    problem_type="regression",
    quantization_config=bnb,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
)
model.config.pad_token_id = tok.pad_token_id
print(model.config.problem_type, "| labels:", model.config.num_labels)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    # Without this the randomly-initialised regression head is never
    # trained and the model cannot learn anything, however long it runs.
    modules_to_save=["score"],
)

model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 4. DatasetThe prompt is the same headline block the live sentiment node builds — a3-day window, newest first — so the model trains on exactly the shape itwould see at inference. No instruction wrapper: there is nothing toinstruct, the head outputs a number.

In [ ]:
from torch.utils.data import Dataset

def to_text(row):
    heads = "\n".join(f"- {h}" for h in row["headlines"])
    return f"Symbol: {row['symbol']}\nHeadlines:\n{heads}"

class HeadlineReturns(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        enc = tok(to_text(r), truncation=True, max_length=MAX_LEN)
        enc["labels"] = float(r["label"])
        return enc

from transformers import DataCollatorWithPadding
collator = DataCollatorWithPadding(tok, return_tensors="pt")

ds_train = HeadlineReturns(train)
ds_eval  = HeadlineReturns(val_clean)
print(len(ds_train), "train |", len(ds_eval), "eval")
print()
print(to_text(train[0])[:300])

## 5. TrainTwo epochs. More would overfit hard: the TF-IDF baseline reached in-sampleR² +0.66 and out-of-sample −0.18 on this data, so memorisation is easy andgeneralisation is the problem. If there is signal here, two passes willshow it; if there is not, ten will not create any.

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="/kaggle/working/qwen-return-lora",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,      # effective batch 32
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    bf16=True,
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_eval,
    data_collator=collator,
)

trainer.train()

## 6. Evaluate — the same metrics as the baselineIdentical to `scripts/tfidf_baseline.py`, deliberately, so the two numbersare comparable rather than merely both present.R² is computed against the **training** mean, not the validation mean,because at inference time you do not know the future mean. Spearman IC isthe metric that actually matters in this field; R² on fat-tailed returnsis dominated by a handful of outliers.

In [ ]:
import numpy as np

TRAIN_MEAN = float(np.mean([r["label"] for r in train]))

def spearman(a, b):
    def ranks(x):
        order = np.argsort(x, kind="mergesort")
        r = np.empty(len(x), dtype=float); r[order] = np.arange(len(x), dtype=float)
        _, inv, counts = np.unique(x, return_inverse=True, return_counts=True)
        sums = np.zeros(len(counts)); np.add.at(sums, inv, r)
        return (sums / counts)[inv]
    ra, rb = ranks(a) - ranks(a).mean(), ranks(b) - ranks(b).mean()
    d = np.sqrt((ra**2).sum() * (rb**2).sum())
    return float((ra*rb).sum()/d) if d else 0.0

@torch.no_grad()
def predict(rows, bs=16):
    model.eval()
    out = []
    for i in range(0, len(rows), bs):
        batch = [to_text(r) for r in rows[i:i+bs]]
        enc = tok(batch, return_tensors="pt", padding=True,
                  truncation=True, max_length=MAX_LEN).to(model.device)
        out.append(model(**enc).logits.squeeze(-1).float().cpu().numpy())
    return np.concatenate(out)

def report(name, rows):
    y = np.array([r["label"] for r in rows], dtype=float)
    pred = predict(rows)
    const = np.full_like(y, TRAIN_MEAN)
    ss_res = ((y-pred)**2).sum(); ss_tot = ((y-const)**2).sum()
    r2 = 1 - ss_res/ss_tot
    ic = spearman(pred, y)
    up = y > 0
    acc = float(((pred > 0) == up).mean())
    base = float(max(up.mean(), 1-up.mean()))
    print(f"\n=== {name}  (n={len(y):,}) ===")
    print(f"  MAE           {np.abs(y-pred).mean():.4f}   constant {np.abs(y-const).mean():.4f}")
    print(f"  R^2           {r2:+.4f}   {'BEATS' if r2>0 else 'WORSE THAN'} a constant")
    print(f"  Spearman IC   {ic:+.4f}")
    print(f"  sign accuracy {acc:.3f}     always-majority {base:.3f}")
    print(f"  pred spread   sd {pred.std():.4f}  min {pred.min():+.3f}  max {pred.max():+.3f}")
    return ic

report("val_time — later dates, seen symbols", val_time)
report("val_symbol — LEAKY, overlapping dates", val_symbol)
ic_clean = report("val_clean — unseen symbols AND dates  << DECIDES", val_clean)

## 7. The verdictCompared against the bar written down before training, not one chosen now.Watch the **prediction spread** printed above as well as the IC. If thestandard deviation is near zero the model has collapsed to predicting themean — the regression equivalent of v1's always-HOLD, and it would producea respectable MAE while having learned nothing.

In [ ]:
BAR = 0.03          # pre-registered
TFIDF_IC = 0.0024   # scripts/tfidf_baseline.py, same split

print(f"TF-IDF baseline IC   {TFIDF_IC:+.4f}")
print(f"this model IC        {ic_clean:+.4f}")
print(f"pre-registered bar   {BAR:+.4f}")
print()
if ic_clean >= BAR and ic_clean > TFIDF_IC:
    print("PASS — clears the bar and beats the baseline.")
    print("Next: plug it into sentiment_analyst.py behind the swappable flag,")
    print("then re-ablate the backtest to see whether it helps the SYSTEM.")
    print("Clearing a modelling bar is not the same as improving the trading result;")
    print("the score node in v1 cleared its own bar and still traded worse.")
else:
    print("FAIL — report it and stop.")
    print("Headlines do not predict 5-day excess returns at a level this setup")
    print("can detect, on 104 symbols over 19 months, with a leak-free split.")
    print("That is a real finding about the task, properly measured. It is not")
    print("a reason to widen the search until something looks significant.")

In [ ]:
model.save_pretrained("/kaggle/working/qwen-return-lora-final")
tok.save_pretrained("/kaggle/working/qwen-return-lora-final")
print("saved — download this directory even on a FAIL; the negative result")
print("is only reproducible if the weights that produced it still exist.")